# Company Alias Dimension Loader

This notebook maintains the `warehouse.dim_company_alias` dimension table.

**Purpose**: Map company name variations to canonical company entities

**Key Features**:
* Links company aliases to canonical companies via company_sk
* Tracks match confidence and method
* Enables fuzzy company name lookup

**Source**: `semantic.sem_company_canonical`

**Target**: `workspace.warehouse.dim_company_alias`

In [0]:
%sql
-- Extract all company name variations and their canonical mappings
CREATE OR REPLACE TEMP VIEW alias_extract AS
SELECT 
  c.company_name_norm as alias_name,
  c.canonical_company_name,
  c.company_match_method,
  c.company_match_confidence,
  c.active_flag as is_active
FROM workspace.semantic.sem_company_canonical c
WHERE c.company_name_norm IS NOT NULL
  AND c.canonical_company_name IS NOT NULL;

In [0]:
%sql
-- Join to company dimension to get stable company_sk
CREATE OR REPLACE TEMP VIEW alias_with_keys AS
SELECT
  COALESCE(da.company_alias_sk, ROW_NUMBER() OVER (ORDER BY a.alias_name, a.canonical_company_name) + COALESCE(max_sk, 0)) as company_alias_sk,
  a.alias_name,
  a.canonical_company_name,
  dc.company_sk,
  a.company_match_method,
  a.company_match_confidence as match_confidence,
  COALESCE(a.is_active, TRUE) as is_active,
  CURRENT_TIMESTAMP() as updated_at
FROM alias_extract a
INNER JOIN workspace.warehouse.dim_company dc
  ON a.canonical_company_name = dc.company_name_canonical
LEFT JOIN workspace.warehouse.dim_company_alias da
  ON a.alias_name = da.alias_name
  AND a.canonical_company_name = da.canonical_company_name
CROSS JOIN (
  SELECT COALESCE(MAX(company_alias_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_company_alias
) max_keys;

In [0]:
%sql
-- Merge into target dimension
MERGE INTO workspace.warehouse.dim_company_alias target
USING alias_with_keys source
ON target.alias_name = source.alias_name
  AND target.canonical_company_name = source.canonical_company_name
WHEN MATCHED THEN UPDATE SET
  target.company_sk = source.company_sk,
  target.company_match_method = source.company_match_method,
  target.match_confidence = source.match_confidence,
  target.is_active = source.is_active,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  company_alias_sk,
  alias_name,
  canonical_company_name,
  company_sk,
  company_match_method,
  match_confidence,
  is_active,
  updated_at
) VALUES (
  source.company_alias_sk,
  source.alias_name,
  source.canonical_company_name,
  source.company_sk,
  source.company_match_method,
  source.match_confidence,
  source.is_active,
  source.updated_at
);

In [0]:
%sql
-- Validate company alias dimension
SELECT 
  COUNT(*) as total_aliases,
  COUNT(DISTINCT company_sk) as distinct_companies,
  COUNT(DISTINCT alias_name) as distinct_alias_names,
  AVG(match_confidence) as avg_confidence,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_aliases
FROM workspace.warehouse.dim_company_alias;

-- Sample aliases
SELECT 
  company_alias_sk,
  alias_name,
  canonical_company_name,
  company_sk,
  match_confidence,
  is_active
FROM workspace.warehouse.dim_company_alias
ORDER BY company_sk, alias_name
LIMIT 20;